In [ ]:
API_KEY = "여기에_본인_ECOS_API_KEY"

# 1) 통계표코드 후보 찾기
candidates = find_stat_table_code(API_KEY, keyword="수출물량지수", lang="kr")

print("통계표코드 후보(상위 10개):")
for c in candidates[:10]:
    print("-", c[0], c[1])

# 2) 보통 1순위 후보로 시도 (안 되면 2~3개를 바꿔가며 테스트)
stat_code = candidates[0][0]

# 3) 반도체 / 운송장비 수출물량지수 다운로드
semi = fetch_series_statistic_search(
    API_KEY,
    stat_code=stat_code,
    item_code="3091AA",          # 반도체
    cycle="M",
    start_date="201001",
    end_date="202512",
    lang="kr"
)

transport = fetch_series_statistic_search(
    API_KEY,
    stat_code=stat_code,
    item_code="3122AA",          # 운송장비
    cycle="M",
    start_date="201001",
    end_date="202512",
    lang="kr"
)

# 4) 합치기(와이드 형태)
merged = (
    semi[["date", "value"]].rename(columns={"value": "semi_export_volume_index"})
    .merge(
        transport[["date", "value"]].rename(columns={"value": "transport_export_volume_index"}),
        on="date",
        how="outer"
    )
    .sort_values("date")
    .reset_index(drop=True)
)

print("\n[반도체] 샘플:")
print(semi.head(5))

print("\n[운송장비] 샘플:")
print(transport.head(5))

# 5) 저장
# semi.to_csv("ecos_semi_export_volume_index.csv", index=False, encoding="utf-8-sig")
# transport.to_csv("ecos_transport_export_volume_index.csv", index=False, encoding="utf-8-sig")
# merged.to_csv("ecos_export_volume_index_merged.csv", index=False, encoding="utf-8-sig")
#
# print("\n저장 완료:")
# print("- ecos_semi_export_volume_index.csv")
# print("- ecos_transport_export_volume_index.csv")
# print("- ecos_export_volume_index_merged.csv")